In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from pydub import AudioSegment
import os

# ==========================
# 固定讀取的資料夾
# ==========================
FOLDER = r"C:\Users\jt725\test_ai_output\filter_session\htdemucs_6s\filter_temp_bb866882"

# 固定音軌名稱
TRACKS = [
    "bass.wav",
    "drums.wav",
    "guitar.wav",
    "other.wav",
    "piano.wav",
    "vocals.wav"
]

class AudioMixer:

    def __init__(self, root):

        self.root = root
        self.root.title("Demucs Audio Mixer")
        self.root.geometry("350x320")

        tk.Label(
            root,
            text="選擇要混音的音軌",
            font=("Arial", 12, "bold")
        ).pack(pady=10)

        self.vars = {}

        frame = tk.Frame(root)
        frame.pack(anchor="w", padx=20)

        for track in TRACKS:

            path = os.path.join(FOLDER, track)

            if os.path.exists(path):

                var = tk.BooleanVar(value=True)

                self.vars[path] = var

                tk.Checkbutton(
                    frame,
                    text=track,
                    variable=var
                ).pack(anchor="w")

        tk.Button(
            root,
            text="輸出 MP3",
            font=("Arial", 11),
            command=self.export
        ).pack(pady=20)

    def export(self):

        selected = []

        for path, var in self.vars.items():
            if var.get():
                selected.append(path)

        if len(selected) == 0:
            messagebox.showerror("錯誤", "請至少勾選一個音軌")
            return

        output = filedialog.asksaveasfilename(
            title="儲存 MP3",
            defaultextension=".mp3",
            filetypes=[("MP3", "*.mp3")]
        )

        if output == "":
            return

        mixed = None

        for wav in selected:

            audio = AudioSegment.from_wav(wav)

            if mixed is None:
                mixed = audio
            else:
                mixed = mixed.overlay(audio)

        mixed.export(output, format="mp3")

        messagebox.showinfo("完成", "MP3 已成功輸出！")


if __name__ == "__main__":

    if not os.path.exists(FOLDER):
        messagebox.showerror("錯誤", f"找不到資料夾：\n{FOLDER}")
        raise SystemExit

    root = tk.Tk()
    app = AudioMixer(root)
    root.mainloop()

In [1]:
#=============================================================
import os
import math
import time
import uuid
import shutil
import threading
import subprocess
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

# =========================================================
# 基本設定：主要只需要改這裡
# =========================================================

# 如果你的 ffmpeg 在這個資料夾，就填 bin 資料夾路徑；不需要就留空字串 ""
FFMPEG_BIN = r"C:\Users\jt725\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin"

# Demucs 輸出總資料夾
OUTPUT_BASE = r"C:\Users\jt725\test_ai_output"

# Demucs 模型名稱
DEMUCS_MODEL = "htdemucs_6s"

# Demucs 會輸出的音軌名稱
TRACK_NAMES = ["bass", "drums", "guitar", "piano", "other", "vocals"]

# 播放器設定
MIXER_RATE = 44100
MIXER_BITS = -16
MIXER_CHANNELS = 2
MIXER_BUFFER = 512


if FFMPEG_BIN and os.path.exists(FFMPEG_BIN):
    os.environ["PATH"] = FFMPEG_BIN + os.pathsep + os.environ.get("PATH", "")

try:
    import pygame
except ImportError:
    pygame = None

try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    AudioSegment = None
    PYDUB_AVAILABLE = False

try:
    import wave
except ImportError:
    wave = None


class DemucsMixerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("AI 音樂分離混音器")
        # 視窗不用開很高，內容太長時可以上下滑動
        self.root.geometry("820x560")
        self.root.minsize(760, 480)

        self.input_path = tk.StringVar(value="")
        self.track_folder = tk.StringVar(value="尚未載入音軌資料夾")
        self.status_text = tk.StringVar(value="請先選擇音樂，或直接讀取已分離好的音軌資料夾。")
        self.device_var = tk.StringVar(value="cuda")
        self.progress_val = tk.DoubleVar(value=0)

        self.is_processing = False
        self.is_exporting = False
        self.demucs_process = None

        self.current_pos = 0.0
        self.start_tick = 0.0
        self.total_duration = 0.0
        self.is_playing = False

        self.tracks = {}
        for name in TRACK_NAMES:
            self.tracks[name] = {
                "var": tk.BooleanVar(value=True),
                "volume": tk.DoubleVar(value=100),
                "file": "",
                "audio_segment": None,
                "audio_sound": None,
            }

        self._init_audio_engine()
        self._build_style()
        self._build_ui()
        self._refresh_track_list_ui()

    # =====================================================
    # UI
    # =====================================================

    def _build_style(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass

        bg = "#f4f6fb"
        card = "#ffffff"
        text = "#1f2937"
        muted = "#6b7280"
        primary = "#2563eb"

        self.root.configure(bg=bg)
        style.configure("TFrame", background=bg)
        style.configure("Card.TFrame", background=card, relief="flat")
        style.configure("TLabel", background=bg, foreground=text, font=("Microsoft JhengHei UI", 10))
        style.configure("Card.TLabel", background=card, foreground=text, font=("Microsoft JhengHei UI", 10))
        style.configure("Muted.TLabel", background=card, foreground=muted, font=("Microsoft JhengHei UI", 9))
        style.configure("Title.TLabel", background=bg, foreground=text, font=("Microsoft JhengHei UI", 18, "bold"))
        style.configure("Subtitle.TLabel", background=bg, foreground=muted, font=("Microsoft JhengHei UI", 10))
        style.configure("Section.TLabel", background=card, foreground=text, font=("Microsoft JhengHei UI", 11, "bold"))
        style.configure("TButton", font=("Microsoft JhengHei UI", 10), padding=(10, 6))
        style.configure("Primary.TButton", font=("Microsoft JhengHei UI", 10, "bold"), padding=(12, 7))
        style.configure("TCheckbutton", background=card, font=("Microsoft JhengHei UI", 10))
        style.configure("TRadiobutton", background=card, font=("Microsoft JhengHei UI", 10))
        style.configure("Horizontal.TProgressbar", thickness=10)

        # ttk 在 Windows 上不一定允許完整改 button 顏色，所以保留系統原生樣式較穩定
        self.colors = {
            "bg": bg,
            "card": card,
            "text": text,
            "muted": muted,
            "primary": primary,
        }

    def _build_ui(self):
        # 外層改成可滑動視窗：Canvas + Scrollbar
        wrapper = ttk.Frame(self.root)
        wrapper.pack(fill="both", expand=True)

        self.canvas = tk.Canvas(
            wrapper,
            bg=self.colors["bg"],
            highlightthickness=0,
            bd=0,
        )
        scrollbar = ttk.Scrollbar(wrapper, orient="vertical", command=self.canvas.yview)
        self.canvas.configure(yscrollcommand=scrollbar.set)

        scrollbar.pack(side="right", fill="y")
        self.canvas.pack(side="left", fill="both", expand=True)

        main = ttk.Frame(self.canvas, padding=18)
        self.canvas_window = self.canvas.create_window((0, 0), window=main, anchor="nw")

        def update_scroll_region(_event=None):
            self.canvas.configure(scrollregion=self.canvas.bbox("all"))

        def resize_content(event):
            self.canvas.itemconfig(self.canvas_window, width=event.width)

        main.bind("<Configure>", update_scroll_region)
        self.canvas.bind("<Configure>", resize_content)

        # 滑鼠滾輪支援 Windows / macOS / Linux
        self.canvas.bind_all("<MouseWheel>", self._on_mousewheel)
        self.canvas.bind_all("<Button-4>", self._on_mousewheel)
        self.canvas.bind_all("<Button-5>", self._on_mousewheel)

        ttk.Label(main, text="AI 音樂分離混音器", style="Title.TLabel").pack(anchor="w")
        ttk.Label(
            main,
            text="整合 Demucs 分離、同步試聽、勾選音軌、輸出 MP3。",
            style="Subtitle.TLabel",
        ).pack(anchor="w", pady=(2, 14))

        self._build_file_card(main)
        self._build_demucs_card(main)
        self._build_player_card(main)
        self._build_tracks_card(main)
        self._build_export_card(main)

        # 狀態列固定在底部，不跟著內容一起滾動
        status_bar = ttk.Frame(self.root, padding=(18, 8))
        status_bar.pack(fill="x", side="bottom")
        ttk.Label(status_bar, textvariable=self.status_text, style="Subtitle.TLabel").pack(anchor="w")

    def _on_mousewheel(self, event):
        # Windows / macOS
        if hasattr(event, "delta") and event.delta:
            self.canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
        # Linux
        elif getattr(event, "num", None) == 4:
            self.canvas.yview_scroll(-3, "units")
        elif getattr(event, "num", None) == 5:
            self.canvas.yview_scroll(3, "units")

    def _card(self, parent, title):
        outer = tk.Frame(parent, bg=self.colors["card"], bd=0, highlightthickness=1, highlightbackground="#e5e7eb")
        outer.pack(fill="x", pady=7)
        inner = ttk.Frame(outer, style="Card.TFrame", padding=12)
        inner.pack(fill="both", expand=True)
        ttk.Label(inner, text=title, style="Section.TLabel").pack(anchor="w", pady=(0, 8))
        return inner

    def _build_file_card(self, parent):
        card = self._card(parent, "1. 選擇音樂 / 讀取音軌")

        row = ttk.Frame(card, style="Card.TFrame")
        row.pack(fill="x")
        entry = ttk.Entry(row, textvariable=self.input_path)
        entry.pack(side="left", fill="x", expand=True)
        ttk.Button(row, text="選取音樂", command=self.select_input_file).pack(side="left", padx=(8, 0))

        row2 = ttk.Frame(card, style="Card.TFrame")
        row2.pack(fill="x", pady=(8, 0))
        ttk.Label(row2, textvariable=self.track_folder, style="Muted.TLabel").pack(side="left", fill="x", expand=True)
        ttk.Button(row2, text="讀取已分離資料夾", command=self.select_track_folder).pack(side="left", padx=(8, 0))

    def _build_demucs_card(self, parent):
        card = self._card(parent, "2. AI 樂器分離")

        row = ttk.Frame(card, style="Card.TFrame")
        row.pack(fill="x")
        ttk.Radiobutton(row, text="GPU / CUDA", variable=self.device_var, value="cuda").pack(side="left", padx=(0, 18))
        ttk.Radiobutton(row, text="CPU", variable=self.device_var, value="cpu").pack(side="left")
        self.btn_start_ai = ttk.Button(row, text="啟動 Demucs 分離", style="Primary.TButton", command=self.start_separation_thread)
        self.btn_start_ai.pack(side="right")

        self.pb = ttk.Progressbar(card, variable=self.progress_val, maximum=100, mode="determinate")
        self.pb.pack(fill="x", pady=(10, 2))
        ttk.Label(card, text="如果 CUDA 失敗，可以切換成 CPU 再試一次。", style="Muted.TLabel").pack(anchor="w")

    def _build_player_card(self, parent):
        card = self._card(parent, "3. 同步試聽")

        self.lbl_time = ttk.Label(card, text="00:00 / 00:00", style="Card.TLabel", font=("Consolas", 14, "bold"))
        self.lbl_time.pack(anchor="center")

        self.play_pb = ttk.Progressbar(card, orient="horizontal", mode="determinate", maximum=100)
        self.play_pb.pack(fill="x", pady=(8, 10))

        row = ttk.Frame(card, style="Card.TFrame")
        row.pack(anchor="center")
        ttk.Button(row, text="⏮ 5 秒", command=lambda: self.seek(-5)).grid(row=0, column=0, padx=5)
        self.btn_play = ttk.Button(row, text="播放", style="Primary.TButton", command=self.toggle_play)
        self.btn_play.grid(row=0, column=1, padx=5)
        ttk.Button(row, text="暫停", command=self.pause_playback).grid(row=0, column=2, padx=5)
        ttk.Button(row, text="停止", command=self.stop_playback).grid(row=0, column=3, padx=5)
        ttk.Button(row, text="5 秒 ⏭", command=lambda: self.seek(5)).grid(row=0, column=4, padx=5)

    def _build_tracks_card(self, parent):
        card = self._card(parent, "4. 音軌選擇與音量")

        toolbar = ttk.Frame(card, style="Card.TFrame")
        toolbar.pack(fill="x", pady=(0, 8))
        ttk.Button(toolbar, text="全選", command=lambda: self.set_all_tracks(True)).pack(side="left")
        ttk.Button(toolbar, text="全部取消", command=lambda: self.set_all_tracks(False)).pack(side="left", padx=6)

        self.track_rows_frame = ttk.Frame(card, style="Card.TFrame")
        self.track_rows_frame.pack(fill="x")

    def _build_export_card(self, parent):
        card = self._card(parent, "5. 輸出混音")

        row = ttk.Frame(card, style="Card.TFrame")
        row.pack(fill="x")
        ttk.Label(
            row,
            text="會把目前勾選的音軌混成一個 MP3。",
            style="Muted.TLabel",
        ).pack(side="left", fill="x", expand=True)
        self.btn_export = ttk.Button(row, text="輸出 MP3", style="Primary.TButton", command=self.start_export_thread)
        self.btn_export.pack(side="right")

    def _refresh_track_list_ui(self):
        for child in self.track_rows_frame.winfo_children():
            child.destroy()

        for row_index, name in enumerate(TRACK_NAMES):
            data = self.tracks[name]
            row = ttk.Frame(self.track_rows_frame, style="Card.TFrame")
            row.pack(fill="x", pady=3)

            exists = bool(data["file"] and os.path.exists(data["file"]))
            display_name = f"{name}.wav"
            state_text = "已載入" if exists else "未載入"

            cb = ttk.Checkbutton(row, text=display_name, variable=data["var"], command=self.on_track_change)
            cb.pack(side="left")

            ttk.Label(row, text=state_text, style="Muted.TLabel", width=8).pack(side="left", padx=(12, 4))

            ttk.Label(row, text="音量", style="Muted.TLabel").pack(side="left", padx=(8, 4))
            scale = ttk.Scale(
                row,
                from_=0,
                to=150,
                variable=data["volume"],
                command=lambda _v, track=name: self.on_volume_change(track),
            )
            scale.pack(side="left", fill="x", expand=True, padx=4)

            value_label = ttk.Label(row, text=f"{int(data['volume'].get())}%", style="Muted.TLabel", width=5)
            value_label.pack(side="left")
            data["volume_label"] = value_label

    # =====================================================
    # 基本工具
    # =====================================================

    def _init_audio_engine(self):
        if pygame is None:
            return
        try:
            pygame.mixer.pre_init(MIXER_RATE, MIXER_BITS, MIXER_CHANNELS, MIXER_BUFFER)
            pygame.mixer.init()
            pygame.mixer.set_num_channels(16)
        except Exception:
            # 讓程式可以開，只是在播放時再提醒
            pass

    def ui(self, func, *args, **kwargs):
        self.root.after(0, lambda: func(*args, **kwargs))

    def set_status(self, text):
        self.status_text.set(text)

    def format_time(self, seconds):
        seconds = max(0, int(seconds))
        m, s = divmod(seconds, 60)
        return f"{m:02d}:{s:02d}"

    def get_selected_tracks(self):
        selected = []
        for name, data in self.tracks.items():
            path = data.get("file", "")
            if data["var"].get() and path and os.path.exists(path):
                selected.append((name, path))
        return selected

    def set_all_tracks(self, value):
        for data in self.tracks.values():
            data["var"].set(value)
        self.on_track_change()

    def _get_audio_duration(self, path):
        if PYDUB_AVAILABLE:
            try:
                return len(AudioSegment.from_file(path)) / 1000.0
            except Exception:
                pass

        if wave is not None and path.lower().endswith(".wav"):
            try:
                with wave.open(path, "rb") as f:
                    return f.getnframes() / float(f.getframerate())
            except Exception:
                pass

        return 0.0

    # =====================================================
    # 檔案與資料夾
    # =====================================================

    def select_input_file(self):
        path = filedialog.askopenfilename(
            title="選擇音樂檔",
            filetypes=[
                ("Audio", "*.wav *.mp3 *.flac *.m4a *.aac *.ogg"),
                ("WAV", "*.wav"),
                ("All files", "*.*"),
            ],
        )
        if not path:
            return

        self.input_path.set(path)
        self.total_duration = self._get_audio_duration(path)
        self.stop_playback()
        self.set_status("已選擇音樂，可以開始 Demucs 分離。")

    def select_track_folder(self):
        folder = filedialog.askdirectory(title="選擇 Demucs 分離後的音軌資料夾")
        if not folder:
            return
        self.load_track_folder(folder)

    def load_track_folder(self, folder):
        found_count = 0
        for name in TRACK_NAMES:
            path = os.path.join(folder, f"{name}.wav")
            self.tracks[name]["file"] = path if os.path.exists(path) else ""
            self.tracks[name]["audio_segment"] = None
            self.tracks[name]["audio_sound"] = None
            if os.path.exists(path):
                found_count += 1

        self.track_folder.set(folder)
        self._preload_audio()
        self._refresh_track_list_ui()
        self.stop_playback()

        if found_count == 0:
            self.set_status("這個資料夾沒有找到 bass.wav / drums.wav / vocals.wav 等音軌。")
            messagebox.showwarning("找不到音軌", "這個資料夾沒有找到 Demucs 音軌 wav 檔。")
        else:
            self.set_status(f"已載入 {found_count} 個音軌，可以播放或輸出 MP3。")

    # =====================================================
    # Demucs 分離
    # =====================================================

    def start_separation_thread(self):
        if self.is_processing:
            return
        if not self.input_path.get():
            messagebox.showwarning("提示", "請先選擇音樂檔。")
            return

        self.stop_playback()
        self.progress_val.set(0)
        t = threading.Thread(target=self.run_demucs, daemon=True)
        t.start()

    def run_demucs(self):
        self.is_processing = True
        self.ui(self.btn_start_ai.config, state="disabled")
        self.ui(self.set_status, f"Demucs 正在使用 {self.device_var.get().upper()} 分離中...")

        input_file = self.input_path.get()
        job_output = os.path.join(OUTPUT_BASE, "filter_session")
        os.makedirs(OUTPUT_BASE, exist_ok=True)

        temp_ext = os.path.splitext(input_file)[1] or ".wav"
        temp_name = f"filter_temp_{uuid.uuid4().hex[:8]}{temp_ext}"
        temp_path = os.path.join(job_output, temp_name)

        try:
            if os.path.exists(job_output):
                shutil.rmtree(job_output)
            os.makedirs(job_output, exist_ok=True)

            shutil.copy2(input_file, temp_path)

            cmd = [
                "demucs",
                "-d",
                self.device_var.get(),
                "-n",
                DEMUCS_MODEL,
                temp_path,
                "-o",
                job_output,
            ]

            creation_flags = 0
            if os.name == "nt":
                creation_flags = subprocess.CREATE_NO_WINDOW

            self.demucs_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                creationflags=creation_flags,
            )

            while self.demucs_process.poll() is None:
                if self.demucs_process.stdout:
                    self.demucs_process.stdout.readline()

                curr = self.progress_val.get()
                if curr < 95:
                    self.ui(self.progress_val.set, curr + 0.4)
                time.sleep(0.1)

            if self.demucs_process.returncode != 0:
                self.ui(self.progress_val.set, 0)
                self.ui(self.set_status, "分離失敗。若使用 CUDA 失敗，請切換 CPU 再試一次。")
                self.ui(messagebox.showerror, "分離失敗", "Demucs 分離失敗。若是 CUDA 錯誤，請改用 CPU。")
                return

            basename = os.path.splitext(os.path.basename(temp_path))[0]
            track_dir = os.path.join(job_output, DEMUCS_MODEL, basename)

            self.ui(self.progress_val.set, 100)
            self.ui(self.load_track_folder, track_dir)
            self.ui(messagebox.showinfo, "成功", "樂器分離完成！")

        except FileNotFoundError:
            self.ui(self.set_status, "找不到 demucs 指令，請確認已安裝 demucs，且 Python 環境正確。")
            self.ui(messagebox.showerror, "找不到 Demucs", "找不到 demucs 指令。\n請先安裝：pip install demucs")
        except Exception as e:
            self.ui(self.set_status, f"錯誤：{e}")
            self.ui(messagebox.showerror, "錯誤", str(e))
        finally:
            try:
                if os.path.exists(temp_path):
                    os.remove(temp_path)
            except Exception:
                pass
            self.is_processing = False
            self.ui(self.btn_start_ai.config, state="normal")

    # =====================================================
    # 音訊預載與播放
    # =====================================================

    def _preload_audio(self):
        if pygame is None:
            return

        first_duration = 0.0
        for name, data in self.tracks.items():
            path = data.get("file", "")
            data["audio_segment"] = None
            data["audio_sound"] = None

            if not path or not os.path.exists(path):
                continue

            try:
                if PYDUB_AVAILABLE:
                    seg = AudioSegment.from_file(path)
                    data["audio_segment"] = seg
                    if first_duration == 0:
                        first_duration = len(seg) / 1000.0

                data["audio_sound"] = pygame.mixer.Sound(path)
            except Exception:
                data["audio_sound"] = None

        if first_duration > 0:
            self.total_duration = first_duration
        else:
            for _name, path in self.get_selected_tracks():
                self.total_duration = self._get_audio_duration(path)
                if self.total_duration > 0:
                    break

        self.update_ui_display(self.current_pos)

    def toggle_play(self):
        if self.is_playing:
            self.pause_playback()
        else:
            self.start_playback()

    def start_playback(self):
        if pygame is None:
            messagebox.showerror("缺少套件", "尚未安裝 pygame。\n請先執行：pip install pygame")
            return

        selected = self.get_selected_tracks()
        if len(selected) == 0:
            messagebox.showwarning("提示", "請至少勾選一個已載入的音軌。")
            return

        try:
            pygame.mixer.stop()
        except Exception:
            pass

        start_ms = int(self.current_pos * 1000)
        prepared = []

        for name, _path in selected:
            data = self.tracks[name]
            sound = None

            if PYDUB_AVAILABLE and start_ms > 0 and data.get("audio_segment") is not None:
                try:
                    seg = data["audio_segment"][start_ms:]
                    seg = seg.set_frame_rate(MIXER_RATE).set_channels(MIXER_CHANNELS).set_sample_width(2)
                    sound = pygame.mixer.Sound(buffer=seg.raw_data)
                except Exception:
                    sound = data.get("audio_sound")
            else:
                sound = data.get("audio_sound")

            if sound:
                volume = max(0, min(float(data["volume"].get()) / 100.0, 1.5))
                prepared.append((sound, volume))

        if len(prepared) == 0:
            messagebox.showwarning("提示", "音軌尚未成功載入，請重新分離或重新選擇音軌資料夾。")
            return

        for index, (sound, volume) in enumerate(prepared):
            channel = pygame.mixer.Channel(index)
            channel.set_volume(min(volume, 1.0))  # pygame 建議 0.0 ~ 1.0，超過 100% 只在輸出 MP3 時保留
            channel.play(sound)

        self.is_playing = True
        self.start_tick = time.time()
        self.btn_play.config(text="播放中")
        self.run_sync_loop()

    def pause_playback(self):
        if not self.is_playing:
            return
        self.current_pos += time.time() - self.start_tick
        self.current_pos = max(0, min(self.current_pos, self.total_duration))
        if pygame is not None:
            pygame.mixer.stop()
        self.is_playing = False
        self.btn_play.config(text="播放")
        self.update_ui_display(self.current_pos)

    def stop_playback(self):
        if pygame is not None:
            try:
                pygame.mixer.stop()
            except Exception:
                pass
        self.is_playing = False
        self.current_pos = 0.0
        if hasattr(self, "btn_play"):
            self.btn_play.config(text="播放")
        if hasattr(self, "play_pb"):
            self.update_ui_display(0)

    def seek(self, seconds):
        if self.is_playing:
            self.current_pos += time.time() - self.start_tick

        self.current_pos += seconds
        self.current_pos = max(0, min(self.current_pos, self.total_duration))

        if self.is_playing:
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def on_track_change(self):
        if self.is_playing:
            self.current_pos += time.time() - self.start_tick
            self.current_pos = max(0, min(self.current_pos, self.total_duration))
            self.start_playback()
        else:
            self.update_ui_display(self.current_pos)

    def on_volume_change(self, track):
        data = self.tracks[track]
        if "volume_label" in data:
            data["volume_label"].config(text=f"{int(float(data['volume'].get()))}%")

    def run_sync_loop(self):
        if not self.is_playing:
            return

        elapsed = time.time() - self.start_tick
        now_pos = self.current_pos + elapsed

        if self.total_duration > 0 and now_pos >= self.total_duration:
            self.stop_playback()
            return

        self.update_ui_display(now_pos)
        self.root.after(50, self.run_sync_loop)

    def update_ui_display(self, pos):
        if not hasattr(self, "lbl_time"):
            return

        self.lbl_time.config(text=f"{self.format_time(pos)} / {self.format_time(self.total_duration)}")
        if self.total_duration > 0:
            self.play_pb["value"] = max(0, min((pos / self.total_duration) * 100, 100))
        else:
            self.play_pb["value"] = 0

    # =====================================================
    # MP3 輸出
    # =====================================================

    def start_export_thread(self):
        if self.is_exporting:
            return

        selected = self.get_selected_tracks()
        if len(selected) == 0:
            messagebox.showwarning("提示", "請至少勾選一個已載入的音軌。")
            return

        if not PYDUB_AVAILABLE:
            messagebox.showerror("缺少套件", "輸出 MP3 需要 pydub。\n請先執行：pip install pydub")
            return

        output = filedialog.asksaveasfilename(
            title="儲存混音 MP3",
            defaultextension=".mp3",
            filetypes=[("MP3", "*.mp3")],
        )
        if not output:
            return

        self.stop_playback()
        t = threading.Thread(target=self.export_mp3, args=(selected, output), daemon=True)
        t.start()

    def export_mp3(self, selected, output):
        self.is_exporting = True
        self.ui(self.btn_export.config, state="disabled")
        self.ui(self.set_status, "正在輸出 MP3...")

        try:
            mixed = None

            for name, path in selected:
                audio = AudioSegment.from_file(path)
                volume_percent = max(0, float(self.tracks[name]["volume"].get()))

                if volume_percent <= 0:
                    continue
                if volume_percent != 100:
                    gain_db = 20 * math.log10(volume_percent / 100)
                    audio = audio.apply_gain(gain_db)

                if mixed is None:
                    mixed = audio
                else:
                    mixed = mixed.overlay(audio)

            if mixed is None:
                self.ui(messagebox.showwarning, "提示", "所有音軌音量都是 0%，無法輸出。")
                return

            mixed.export(output, format="mp3")
            self.ui(self.set_status, f"MP3 已成功輸出：{output}")
            self.ui(messagebox.showinfo, "完成", "MP3 已成功輸出！")

        except Exception as e:
            self.ui(self.set_status, f"輸出失敗：{e}")
            self.ui(messagebox.showerror, "輸出失敗", str(e))
        finally:
            self.is_exporting = False
            self.ui(self.btn_export.config, state="normal")


if __name__ == "__main__":
    root = tk.Tk()
    app = DemucsMixerApp(root)
    root.mainloop()


C:\Users\jt725\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
